<a href="https://colab.research.google.com/github/Innovatewithapple/Resume_JD_AnalysisTransfomer/blob/main/notebook6d8f293bc2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Run this cell only once when start working on project with new session. this cell is just for error fixing of torchao
!pip uninstall -y torchao
import os
os.kill(os.getpid(), 9)

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
!pip show torchao

In [ ]:
!pip install -q transformers datasets peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 113.6 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which

In [ ]:
import torch
import peft
import transformers
print("peft:", peft.__version__)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)

peft: 0.18.1
transformers: 5.0.0
torch: 2.10.0+cu128


In [ ]:
import torch
import random
import numpy as np
import os

def setupSeed(seed=32):
  os.environ['PYTHONHASHSEED'] = str(seed)
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True

setupSeed()

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from datasets import load_dataset
from peft import LoraConfig,get_peft_model
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from datasets import Dataset as HFDataset
from transformers import AutoTokenizer,AutoModel,AutoModelForSequenceClassification,DataCollatorWithPadding
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
import gc
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
#-------Auto Tokenizer--------!
autoToken = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')

#--------Model-------------!
encoder_model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-v3-base",num_labels=3)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight      

In [ ]:
#-------Lora Configuration-----!
lora_config = LoraConfig(r=16,lora_alpha=32,target_modules=['query_proj','value_proj'],lora_dropout=0.1,bias='none')

#------Apply on model-------!
model = get_peft_model(encoder_model,lora_config)

In [ ]:
ds = load_dataset("cnamuangtoun/resume-job-description-fit")

train.csv:   0%|          | 0.00/53.4M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/15.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6241 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1759 [00:00<?, ? examples/s]

In [ ]:
resume_texts = ds['train']['resume_text']
jd_texts = ds['train']['job_description_text']
label_text = ds['train']['label']

In [ ]:
val_resume_texts = ds['test']['resume_text']
val_jd_texts = ds['test']['job_description_text']
val_label_text = ds['test']['label']

In [ ]:
labelencoder = LabelEncoder()
label_encoded_text = labelencoder.fit_transform(label_text)
val_label_encoded_text = labelencoder.transform(val_label_text)

In [ ]:
#--------Data Loader--------!
train_Ds = HFDataset.from_dict({
    'resume':resume_texts,
    'jd': jd_texts,
    'label': label_encoded_text
})

val_Ds = HFDataset.from_dict({
    'resume':val_resume_texts,
    'jd': val_jd_texts,
    'label': val_label_encoded_text
})

In [ ]:
model.print_trainable_parameters()

trainable params: 589,824 || all params: 185,014,275 || trainable%: 0.3188


In [ ]:
#-------Tokenize-BatchWise-------!
def tokenize_function(batch):
  return autoToken(batch['resume'],batch['jd'],truncation=True,max_length=512)

In [ ]:
#----------we mapped using huggingface-------!
train_Ds = train_Ds.map(tokenize_function,batched=True,batch_size=32)
val_Ds = val_Ds.map(tokenize_function,batched=True,batch_size=32)

Map:   0%|          | 0/6241 [00:00<?, ? examples/s]

Map:   0%|          | 0/1759 [00:00<?, ? examples/s]

In [ ]:
#-----Remove resume and jd columns from dataset, as now we have input ids and attentionmask in it-------!
train_Ds = train_Ds.remove_columns(['resume','jd'])
val_Ds = val_Ds.remove_columns(['resume','jd'])

train_Ds = train_Ds.rename_column('label','labels')
val_Ds = val_Ds.rename_column('label','labels')

train_Ds.set_format(type="torch",columns=["input_ids", "attention_mask", "labels"])

val_Ds.set_format(type="torch",columns=["input_ids", "attention_mask", "labels"])

In [ ]:
#-----Dynamic padding (huge ram saver)------!
data_collector = DataCollatorWithPadding(tokenizer=autoToken)

# Batch 1
# Longest sample = 280 tokens
# Pad only to 280

# Batch 2
# Longest sample = 410 tokens
# Pad only to 410

In [ ]:
train_loader = DataLoader(dataset=train_Ds,batch_size=4,collate_fn=data_collector,shuffle=True,pin_memory=True,num_workers=0)
val_loader = DataLoader(dataset=val_Ds,batch_size=4,collate_fn=data_collector,shuffle=False,pin_memory=True,num_workers=0)

In [ ]:
classes = np.unique(label_encoded_text)
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=label_encoded_text
)

print(weights)

[1.34911371 0.66189416 1.33697515]


In [ ]:
# MULTI GPU SUPPORT
# -----------------------------------
# if torch.cuda.device_count() > 1:
#     print("Using Multiple GPUs")
#     model = nn.DataParallel(model)

# Move FINAL model to device
model = model.to(device)
optimizer = optim.AdamW(params=filter(lambda p: p.requires_grad, model.parameters()),lr=1e-3,weight_decay=1e-2)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)
# loss_fn = nn.CrossEntropyLoss()
scalar = torch.amp.GradScaler('cuda')

In [ ]:
epochs = 5
accumulation_steps = 8

for epoch in range(epochs):
    model.train()

    train_loss = 0
    train_Correct = 0
    train_total = 0

    optimizer.zero_grad()

    progress_bar_train = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for step, batch in enumerate(progress_bar_train):

        text = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with torch.amp.autocast('cuda'):
            output = model(
                input_ids=text,
                attention_mask=attention_mask
            )

            logits = output.logits

            loss = loss_fn(logits, labels)

            # Gradient accumulation
            loss = loss / accumulation_steps

        scalar.scale(loss).backward()

        if (step + 1) % accumulation_steps == 0:
            scalar.step(optimizer)
            scalar.update()
            optimizer.zero_grad()

        train_loss += loss.item() * accumulation_steps

        pred = torch.argmax(logits, dim=1)

        train_Correct += (pred == labels).sum().item()
        train_total += labels.size(0)

    # Handle leftover batches at end of epoch
    if (step + 1) % accumulation_steps != 0:
        scalar.step(optimizer)
        scalar.update()
        optimizer.zero_grad()

    train_accuracy = train_Correct / train_total
    train_losses = train_loss / len(train_loader)

    # ---------------- Validation ----------------

    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    progress_bar_val = tqdm(val_loader, desc='Validation')

    with torch.no_grad():

        for batch in progress_bar_val:

            text = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            with torch.amp.autocast('cuda'):
                output = model(input_ids=text,attention_mask=attention_mask)

                logits = output.logits

                loss = loss_fn(logits, labels)

            val_loss += loss.item()

            pred = torch.argmax(logits, dim=1)

            val_correct += (pred == labels).sum().item()
            val_total += labels.size(0)

    val_accuracy = val_correct / val_total
    val_losses = val_loss / len(val_loader)

    print(f'\nEpochs: {epoch+1}/{epochs}...')
    print(f'Train_acc: {train_accuracy} | Train_loss: {train_losses}')
    print(f'Val_acc: {val_accuracy} | Val_loss: {val_losses}')
    print('==============================================================')
    gc.collect()
    torch.cuda.empty_cache()

Validation: 100%|██████████| 440/440 [00:40<00:00, 10.74it/s]



Epochs: 1/5...
Train_acc: 0.4786091972440314 | Train_loss: 1.0966198148825168
Val_acc: 0.4872086412734508 | Val_loss: 1.076395579088818


Validation: 100%|██████████| 440/440 [00:40<00:00, 10.75it/s]



Epochs: 2/5...
Train_acc: 0.49879826950809164 | Train_loss: 1.0959297109306967
Val_acc: 0.4872086412734508 | Val_loss: 1.0757398853247815


Validation: 100%|██████████| 440/440 [00:40<00:00, 10.78it/s]



Epochs: 3/5...
Train_acc: 0.4944720397372216 | Train_loss: 1.0960773070373266
Val_acc: 0.4872086412734508 | Val_loss: 1.0780822921882975


Validation: 100%|██████████| 440/440 [00:40<00:00, 10.74it/s]



Epochs: 4/5...
Train_acc: 0.5012017304919083 | Train_loss: 1.0951645227291125
Val_acc: 0.4872086412734508 | Val_loss: 1.0772193976423956


Validation: 100%|██████████| 440/440 [00:41<00:00, 10.73it/s]



Epochs: 5/5...
Train_acc: 0.49815734657907385 | Train_loss: 1.096056732743463
Val_acc: 0.4872086412734508 | Val_loss: 1.0757665401155299


In [ ]:
  gc.collect()
  torch.cuda.empty_cache()

In [ ]:
# Check actual distribution
import pandas as pd
df_train = pd.DataFrame(ds['train'])
print(df_train['label'].value_counts())
print(df_train['label'].value_counts(normalize=True))